# Representation Bias, Fairness Metrics, and Subgroup Error Lab


## Purpose

This lab evaluates subgroup representation, prediction error, and statistical parity difference.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "group": rng.choice(["A", "B"], size=n, p=[0.82, 0.18]),
    "feature_signal": rng.normal(0, 1, size=n),
    "measurement_quality": rng.choice(
        ["high", "medium", "low"],
        size=n,
        p=[0.55, 0.30, 0.15],
    ),
})

data["measurement_error_sd"] = np.where(data["group"] == "B", 0.45, 0.20)

data["measured_feature"] = (
    data["feature_signal"]
    + rng.normal(0, data["measurement_error_sd"], size=n)
)

data.head()

In [ ]:
logit = -0.10 + 1.20 * data["feature_signal"] + np.where(data["group"] == "B", -0.15, 0)
probability = 1 / (1 + np.exp(-logit))
data["true_label"] = rng.binomial(1, probability)

missing_probability = np.where(data["group"] == "B", 0.20, 0.06)
data["measured_feature_missing"] = rng.binomial(1, missing_probability)
data.loc[data["measured_feature_missing"] == 1, "measured_feature"] = np.nan

imputed_feature = data["measured_feature"].fillna(data["measured_feature"].mean())
score = 1 / (1 + np.exp(-(-0.05 + 1.05 * imputed_feature)))

data["predicted_label"] = (score >= 0.50).astype(int)
data["prediction_error"] = (data["predicted_label"] != data["true_label"]).astype(int)

group_summary = (
    data
    .groupby("group")
    .agg(
        units=("unit_id", "count"),
        share=("unit_id", lambda x: len(x) / len(data)),
        positive_prediction_rate=("predicted_label", "mean"),
        error_rate=("prediction_error", "mean"),
        missing_rate=("measured_feature_missing", "mean"),
    )
    .reset_index()
)

rate_a = group_summary.loc[group_summary["group"] == "A", "positive_prediction_rate"].iloc[0]
rate_b = group_summary.loc[group_summary["group"] == "B", "positive_prediction_rate"].iloc[0]

statistical_parity_difference = rate_a - rate_b

group_summary, statistical_parity_difference

## Interpretation

Aggregate performance can hide subgroup differences in representation, missingness, positive prediction rates, and error.